# Lab 03: Random numbers, seeding, and reproducible simulation

- **Generate numbers. Then generate them again.**
- **Keep track of what you ran.**

Open your own copy so you can change the code. From the lab page, use the rocket
menu to open DataHub or Colab. In Colab, choose **File → Save a copy in Drive**,
then click **Connect**. You don't need a GPU.

We'll use NumPy to generate random numbers and Matplotlib to see what they do.
Run the cells from the top. When we repeat a calculation, pay attention to which
cells we rerun.

## Get the packages ready

- **Import before using a package.**

In Colab or your own local notebook, if an import below gives `ModuleNotFoundError`,
remove the `#` from this install command and run it. If you're asked to restart
the kernel or runtime, do that, then run the imports again. Put the `#` back
afterward so Run All doesn't repeat the installation.

On DataHub, these packages should be in the course environment. If they're missing,
check which environment you selected and ask me before installing over it.

In [ ]:
# %pip install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Make a random number generator

- **The seed sets the starting point.**
- **The generator keeps its place.**

These are **pseudorandom** numbers: an algorithm produces a sequence that we use
as random samples. A seed initializes that algorithm. It isn't a measure of how
random or accurate a result is.

`default_rng` creates a generator. We'll name it `rng` and ask it for numbers.
Keep the seed fixed for now.

In [ ]:
seed = 20261006
rng = np.random.default_rng(seed)

- **Five numbers between zero and one.**

`random(5)` returns an array of five floats in $[0, 1)$: zero is included, one is
excluded. This is uniform sampling; equal-width intervals have equal probability.

In [ ]:
first = rng.random(5)
first

- **Same generator. Next five numbers.**

Predict what happens before running this cell. We haven't created a new generator.

In [ ]:
second = rng.random(5)
second

### Start again from the seed

- **Drawing advances the state.**
- **Recreating the generator starts it over.**

Run only the previous cell again. You get another draw because `rng` remembers
where it is in the sequence. Now create a new generator with the original seed.

In [ ]:
replay_rng = np.random.default_rng(seed)
replayed = replay_rng.random(5)
replayed

- **Compare every entry.**

With the same environment and calls, the same starting state reproduces the same
values. Here exact equality is the check we want.

In [ ]:
np.array_equal(first, replayed)

- **Rerun the draw cell. Then rerun the setup too.**

If that comparison is `False`, rerun the cell that creates `rng`, then the cell
that defines `first`, then the two replay cells. Running `first = rng.random(5)`
twice without resetting `rng` replaces `first` with a later part of the sequence.

Change `seed` and repeat that sequence of cells. You should reproduce the new
sequence. Omitting the seed with `np.random.default_rng()` asks NumPy to initialize
from fresh system entropy; it doesn't give you a seed you've written down.

## Choose the distribution

- **Choose the values your model needs.**
- **Check the arguments.**

A generator can sample different distributions. We'll use a separate generator
for these examples so running them doesn't advance the one above.

In [ ]:
distribution_rng = np.random.default_rng(31415)

### Uniform numbers over an interval

- **Set the lower and upper limits.**

Here the target interval runs from $-1$ to $1$. `size=6` asks for six samples.

In [ ]:
distribution_rng.uniform(low=-1.0, high=1.0, size=6)

### Integers

- **The upper limit is excluded.**

For a six-sided die, use `low=1` and `high=7`. Try changing `size`, then change
`high` to `6` and explain which face disappears.

In [ ]:
distribution_rng.integers(low=1, high=7, size=12)

### Normal numbers

- **`loc` is the mean.**
- **`scale` is the standard deviation.**

The normal distribution is the familiar bell curve. `scale` is its width, not its
variance. Here we ask for a distribution centered at zero with standard deviation
one. A finite sample won't have exactly those values.

In [ ]:
normal_samples = distribution_rng.normal(loc=0.0, scale=1.0, size=5000)
normal_samples[:8]

- **Measure the sample we actually drew.**

`ddof=1` uses the sample-variance denominator $N-1$ when estimating the spread.
This standard deviation describes the spread of individual draws.

In [ ]:
print("Sample mean:", np.mean(normal_samples))
print("Sample standard deviation:", np.std(normal_samples, ddof=1))

- **A histogram fluctuates.**
- **Compare its shape with the distribution.**

`density=True` makes the total histogram area one, so we can compare it with the
normal probability density. Bin heights are densities, not probabilities by
themselves; probability corresponds to area.

In [ ]:
x_grid = np.linspace(-4, 4, 301)
normal_density = np.exp(-x_grid**2 / 2) / np.sqrt(2 * np.pi)
fig_normal, ax_normal = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax_normal.hist(normal_samples, bins=35, density=True, alpha=0.6, label="Samples")
ax_normal.plot(x_grid, normal_density, label="Normal density")
ax_normal.set(xlabel="Sample value", ylabel="Probability density")
ax_normal.legend()
plt.show()

- **Fewer samples. Same distribution.**

Change `size=5000` to `size=100` in the normal-sampling cell and rerun it, the
statistics cell, and the plot. Then restore `5000` and rerun all three.
Does a rougher histogram mean
the generator is broken? A smooth histogram alone also doesn't prove a generator
is good; it doesn't tell us whether successive draws are related.

## Find the reseeding mistake

- **Where does the generator get created?**
- **Predict the list before running it.**

This code runs, but it doesn't produce the five successive draws we intended.

In [ ]:
repeated = []
for index in range(5):
    bad_rng = np.random.default_rng(1234)
    repeated.append(bad_rng.random())
repeated

- **Create it once, outside the loop.**

Each iteration above goes back to the same starting state and takes the first
number again. Move the setup outside the loop so the generator can advance.

In [ ]:
good_rng = np.random.default_rng(1234)
draws = []
for index in range(5):
    draws.append(good_rng.random())
draws

## Use the numbers in a simulation

- **Sample points in a square.**
- **Count the points inside a quarter circle.**

Take a square with $0 \leq x < 1$ and $0 \leq y < 1$. The quarter circle of radius
one has area $\pi/4$, and the square has area one. For points sampled uniformly
in the square, the probability of landing inside the quarter circle is $\pi/4$.

That gives us an estimate: four times the fraction inside. We already know
$\pi$, which gives us something to check. This is a Monte Carlo calculation:
we use random samples to estimate a quantity.

In [ ]:
simulation_seed = 271828
n_points = 10000
simulation_rng = np.random.default_rng(simulation_seed)

- **One row per point. Two coordinates per row.**

The tuple `(n_points, 2)` sets the shape of the array. Keep this sampling call
unchanged when we reproduce the run later.

In [ ]:
points = simulation_rng.random((n_points, 2))
points[:5]

- **Separate the two columns.**

In [ ]:
x = points[:, 0]
y = points[:, 1]

- **One `True` or `False` per point.**

A point is inside when $x^2 + y^2 \leq 1$.

In [ ]:
inside = x**2 + y**2 <= 1
inside[:10]

- **The mean of a mask is the fraction that's `True`.**

NumPy counts `True` as one and `False` as zero in this average.

In [ ]:
hit_fraction = np.mean(inside)
pi_estimate = 4 * hit_fraction

- **Look at a subset; calculate with all the points.**

We'll plot the first 600 points to keep the picture readable. The estimate uses
all `n_points` points. `~` flips a Boolean mask.

In [ ]:
shown = 600
shown_inside = inside[:shown]
fig_points, ax_points = plt.subplots(figsize=(5, 5), constrained_layout=True)
ax_points.scatter(x[:shown][shown_inside], y[:shown][shown_inside], s=12, label="Inside")
ax_points.scatter(x[:shown][~shown_inside], y[:shown][~shown_inside], s=12, label="Outside")
circle_x = np.linspace(0, 1, 201)
ax_points.plot(circle_x, np.sqrt(1 - circle_x**2), color="black")
ax_points.set(xlabel="x", ylabel="y", xlim=(0, 1), ylim=(0, 1), aspect="equal")
ax_points.legend()
plt.show()

### Put an uncertainty on it

- **A repeatable result can still fluctuate between seeds.**
- **Report the estimate and its uncertainty together.**

Each point gives a zero or one with probability $p$ of getting one. For $N$
independent points, the variance of their average is $p(1-p)/N$. We don't know
$p$ in a calculation like this, so we estimate it with our hit fraction $\hat p$.
Multiplying the fraction by four also multiplies its standard error by four:

$$
\widehat{\mathrm{SE}}(\hat\pi) = 4\sqrt{\frac{\hat p(1-\hat p)}{N}}.
$$

This is an approximate **standard error of the estimate**, not the spread of
individual points and not a guaranteed bound. It is useful here because we have
independent draws and plenty of points both inside and outside the circle.

In [ ]:
pi_standard_error = 4 * np.sqrt(hit_fraction * (1 - hit_fraction) / n_points)
print(f"seed={simulation_seed}, N={n_points}")
print(f"pi estimate: {pi_estimate:.4f} +/- {pi_standard_error:.4f} (one standard error)")

- **Compare the discrepancy with the uncertainty.**

The difference from `np.pi` is the discrepancy we observe in this run. The
standard error estimates the typical sampling variation. They aren't the same
thing. Even a correct simulation won't always land within one standard error
of the known value.

In [ ]:
difference = pi_estimate - np.pi
print(f"Estimate minus pi: {difference:.4f}")
print(f"Difference / standard error: {difference / pi_standard_error:.2f}")

## Make the run easy to repeat

- **Pass the generator into the function.**
- **Keep the same sampling call.**

Put the calculation into a function so we can change the sample count without
copying the code. The function receives a generator and advances it; it doesn't
choose or reset a seed. It returns the estimate and its standard error.

In [ ]:
def estimate_pi(n_points, rng):
    points = rng.random((n_points, 2))
    inside = points[:, 0]**2 + points[:, 1]**2 <= 1
    fraction = np.mean(inside)
    estimate = 4 * fraction
    standard_error = 4 * np.sqrt(fraction * (1 - fraction) / n_points)
    return estimate, standard_error

- **Recreate the generator. Repeat the calculation.**

Start from the same seed and use the same sample count. If you changed earlier
cells, rerun the simulation from its settings cell through the uncertainty cell
before comparing.

In [ ]:
check_rng = np.random.default_rng(simulation_seed)
check_estimate, check_error = estimate_pi(n_points, check_rng)
print("Same estimate:", check_estimate == pi_estimate)
print("Same uncertainty:", check_error == pi_standard_error)

- **The seed isn't the whole record.**

Adding an extra draw before the simulation advances the state. Changing the
order or shape of the sampling calls can change which values get used. Keep the
code and inputs as well as the seed. NumPy also doesn't guarantee identical
`Generator` results across all versions and environments.

The [NumPy compatibility policy](https://numpy.org/doc/stable/reference/random/compatibility.html)
explains those limits. Matching a previous run checks reproducibility; it doesn't
prove the calculation itself is correct. That's why we also checked the geometry
and compared with a known value.

### Change the sample count

- **Four times the points: about half the standard error.**
- **More points don't guarantee a closer answer in every run.**

Run the calculation at three sample counts. We start a fresh generator with the
same seed for each count, so the smaller samples are contained in the larger
ones here. These are related estimates, not three independent trials.

In [ ]:
for count in [1000, 4000, 16000]:
    count_rng = np.random.default_rng(simulation_seed)
    estimate, standard_error = estimate_pi(count, count_rng)
    print(f"seed={simulation_seed}, N={count:5d}: {estimate:.4f} +/- {standard_error:.4f}")

- **Change the seed, keep the sample count.**

Change `simulation_seed` in the settings cell and rerun from there. Does the
estimate change? Does its uncertainty change much? Don't search for a seed that
gives the closest answer and report only that run.

The uncertainty should fall roughly as $1/\sqrt{N}$ as we increase the sample
count. The discrepancy from the known value can move either way in one run.

## Save what you ran

- **Seed, inputs, code, and environment.**
- **Save the result with its uncertainty.**

We'll write a small JSON file next to the notebook. JSON stores named values in
text, so you can open it later and see what produced the result. Python's `json`
and `platform` modules come with Python; they don't need a separate install.

In [ ]:
import json
import platform
from pathlib import Path

- **Record the generator, not just the seed.**

This records the main simulation run. The generator name identifies the algorithm
used underneath NumPy's sampling methods. Save your edited notebook too: the
file below records inputs and results, but it doesn't contain the code.

In [ ]:
run_record = {
    "seed": simulation_seed,
    "n_points": n_points,
    "sampling_call": "rng.random((n_points, 2))",
    "bit_generator": type(simulation_rng.bit_generator).__name__,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pi_estimate": float(pi_estimate),
    "standard_error": float(pi_standard_error),
}
print(json.dumps(run_record, indent=2))

- **Write it. Read it back.**

Reusing this filename replaces the previous record. Use a different name if you
want to keep several runs. In Colab, download the file from the Files panel before
leaving the runtime.

In [ ]:
record_path = Path("pi_run.json")
record_path.write_text(json.dumps(run_record, indent=2))

In [ ]:
saved_record = json.loads(record_path.read_text())
saved_record == run_record

## Try it with a die

- **Use the same pieces for a new calculation.**
- **Explain what changes when you repeat it.**

Simulate 6000 rolls of a fair six-sided die with a seed you choose.

1. Use `rng.integers` and find the fraction of rolls that are six.
2. Estimate its standard error with $\sqrt{\hat p(1-\hat p)/N}$. There is no
   factor of four for this calculation.
3. Report the seed, number of rolls, fraction, and standard error. Compare the
   result with $1/6$.
4. Recreate the generator with the same seed and confirm you get the same rolls.
5. Try a new seed, then four times as many rolls. Explain which differences you
   expect and which you can't predict for a single run.

Use new names for your variables so this doesn't replace the circle calculation.

In [ ]:
# Create your generator, roll the die, and analyze the results here.

- **Restart. Run from the top.**
- **Check that the saved record matches the run.**

Save your edits, restart the kernel or runtime, and run all cells in order.
Do the replay comparisons still pass? Can someone else tell which seed, sample
count, and code produced your result without asking you?

For other distributions and their arguments, use NumPy's
[random generator reference](https://numpy.org/doc/stable/reference/random/generator.html).